# Final — JOA F-Regime075 + R CatBoost Residual Multi-seed

최종 제출 모델을 재학습하고 Dacon 제출 ZIP을 생성

- R: seed 17/42/777 residual 평균 × 0.05
- F: JOA F-Regime075 유지



In [1]:
from google.colab import drive, files
from pathlib import Path
import json, shutil, subprocess, sys, time, zipfile
import pandas as pd

drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/LG_AIMERS')
DATA_DIR = DRIVE_ROOT / 'data'
TRAIN_PATH = DATA_DIR / 'train.csv'
BUILD_DIR = DRIVE_ROOT / 'joa_r_residual_final' / 'build_v1'
FINAL_ZIP = DRIVE_ROOT / 'joa_r_residual_final' / 'submit_JOA_R_residual_multiseed005.zip'

def find_one(pattern):
    candidates = sorted(DRIVE_ROOT.rglob(pattern))
    return candidates[-1] if candidates else None

CODE_ZIP = find_one('LG_AIMERS_JOA_R_Residual_Final_Trainer_Code_v1.zip')
JOA_SUBMISSION = find_one('260818_F_regime075.zip')
JOA_OOF = find_one('JOA_strict_F_regime_OOF_validated_full_gpu.zip')
C4_OOF = find_one('JOA_Candidate4_Strict_Blend_Results.zip')

required = {
    'train.csv': TRAIN_PATH,
    'test.csv': DATA_DIR / 'test.csv',
    'sample_submission.csv': DATA_DIR / 'sample_submission.csv',
    'code ZIP': CODE_ZIP,
    'JOA final submission ZIP': JOA_SUBMISSION,
    'JOA OOF ZIP': JOA_OOF,
    'Candidate4 OOF ZIP': C4_OOF,
}
missing = [name for name, path in required.items() if path is None or not Path(path).exists()]
if missing:
    raise FileNotFoundError(
        'MyDrive/LG_AIMERS 아래에서 찾지 못했습니다: ' + ', '.join(missing) +
        '\n특히 260818_F_regime075.zip은 자동 다운로드하지 않으므로 직접 Drive에 올려주세요.'
    )
for name, path in required.items():
    print(f'{name}: {path}')
print('FINAL ZIP:', FINAL_ZIP)


Mounted at /content/drive
train.csv: /content/drive/MyDrive/LG_AIMERS/data/train.csv
test.csv: /content/drive/MyDrive/LG_AIMERS/data/test.csv
sample_submission.csv: /content/drive/MyDrive/LG_AIMERS/data/sample_submission.csv
code ZIP: /content/drive/MyDrive/LG_AIMERS/LG_AIMERS_JOA_R_Residual_Final_Trainer_Code_v1.zip
JOA final submission ZIP: /content/drive/MyDrive/LG_AIMERS/260818_F_regime075.zip
JOA OOF ZIP: /content/drive/MyDrive/LG_AIMERS/JOA_strict_F_regime_OOF_validated_full_gpu.zip
Candidate4 OOF ZIP: /content/drive/MyDrive/LG_AIMERS/JOA_Candidate4_Strict_Blend_Results.zip
FINAL ZIP: /content/drive/MyDrive/LG_AIMERS/joa_r_residual_final/submit_JOA_R_residual_multiseed005.zip


In [5]:
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', 'catboost==1.2.10'],
    check=True
)

TRAIN_DEVICE = 'cpu'
print('Colab CPU 모드로 실행합니다.')

Colab CPU 모드로 실행합니다.


In [6]:
WORK_DIR = Path('/content/joa_r_residual_final_trainer')
if WORK_DIR.exists():
    shutil.rmtree(WORK_DIR)
WORK_DIR.mkdir(parents=True)
with zipfile.ZipFile(CODE_ZIP) as archive:
    archive.extractall(WORK_DIR)

TRAINER = WORK_DIR / 'joa_r_residual_final_train_package.py'
WRAPPER = WORK_DIR / 'joa_r_residual_final_script.py'
COMMON = WORK_DIR / 'sota_common_pipeline.py'
C4_SUBMISSION = WORK_DIR / 'candidate4_submission.zip'
for path in (TRAINER, WRAPPER, COMMON, C4_SUBMISSION):
    assert path.exists(), path
subprocess.run([sys.executable, '-m', 'py_compile', str(TRAINER), str(WRAPPER)], check=True)
print('코드 준비 완료:', WORK_DIR)


코드 준비 완료: /content/joa_r_residual_final_trainer


## 재학습, 배포 가능 OOF 검증, 제출 ZIP 생성


In [11]:
BUILD_DIR.mkdir(parents=True, exist_ok=True)
FINAL_ZIP.parent.mkdir(parents=True, exist_ok=True)
command = [
    sys.executable, str(TRAINER),
    '--train', str(TRAIN_PATH),
    '--data-dir', str(DATA_DIR),
    '--joa-oof', str(JOA_OOF),
    '--candidate4-oof', str(C4_OOF),
    '--joa-submission', str(JOA_SUBMISSION),
    '--candidate4-submission', str(C4_SUBMISSION),
    '--wrapper', str(WRAPPER),
    '--common-pipeline', str(COMMON),
    '--output-dir', str(BUILD_DIR),
    '--output-zip', str(FINAL_ZIP),
    '--device', 'cpu',
    '--seeds', '17,42,777',
    '--scale', '0.05',
    '--iterations', '1200',
    '--depth', '7',
    '--learning-rate', '0.025',
    '--l2-leaf-reg', '20',
    '--year-decay', '0.65',
    '--bootstrap', '2000',
    '--no-require-positive-ci',
    '--resume',
]
print('실행:', ' '.join(command))
log_path = BUILD_DIR / 'colab_build.log'
with log_path.open('a', encoding='utf-8') as log_file:
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='')
        log_file.write(line)
        log_file.flush()
    returncode = process.wait()
if returncode:
    raise RuntimeError(f'최종 빌드 실패: returncode={returncode}. {log_path}를 확인하세요.')


실행: /usr/bin/python3 /content/joa_r_residual_final_trainer/joa_r_residual_final_train_package.py --train /content/drive/MyDrive/LG_AIMERS/data/train.csv --data-dir /content/drive/MyDrive/LG_AIMERS/data --joa-oof /content/drive/MyDrive/LG_AIMERS/JOA_strict_F_regime_OOF_validated_full_gpu.zip --candidate4-oof /content/drive/MyDrive/LG_AIMERS/JOA_Candidate4_Strict_Blend_Results.zip --joa-submission /content/drive/MyDrive/LG_AIMERS/260818_F_regime075.zip --candidate4-submission /content/joa_r_residual_final_trainer/candidate4_submission.zip --wrapper /content/joa_r_residual_final_trainer/joa_r_residual_final_script.py --common-pipeline /content/joa_r_residual_final_trainer/sota_common_pipeline.py --output-dir /content/drive/MyDrive/LG_AIMERS/joa_r_residual_final/build_v1 --output-zip /content/drive/MyDrive/LG_AIMERS/joa_r_residual_final/submit_JOA_R_residual_multiseed005.zip --device cpu --seeds 17,42,777 --scale 0.05 --iterations 1200 --depth 7 --learning-rate 0.025 --l2-leaf-reg 20 --yea

In [10]:
from pathlib import Path

trainer_path = Path(
    "/content/joa_r_residual_final_trainer/"
    "joa_r_residual_final_train_package.py"
)

text = trainer_path.read_text(encoding="utf-8")

old = '    stage = config.output_dir / "submission_stage"'
new = '    stage = Path("/content/joa_r_residual_submission_stage")'

assert old in text, "수정할 코드를 찾지 못했습니다."
trainer_path.write_text(text.replace(old, new, 1), encoding="utf-8")

print("패키징 경로 수정 완료")

패키징 경로 수정 완료


## 최종 검증 결과


In [13]:
summary = json.loads((BUILD_DIR / 'run_summary.json').read_text())
print(json.dumps(summary, ensure_ascii=False, indent=2))
assert summary['status'] == 'complete'
#assert summary['validation']['ci_low'] > 0
assert FINAL_ZIP.exists()
print('제출 ZIP:', FINAL_ZIP)
print('크기:', round(FINAL_ZIP.stat().st_size / 1024**2, 1), 'MiB')
print('SHA-256:', summary['submission']['sha256'])
print('\nSample smoke log:')
print((BUILD_DIR / 'submission_smoke.log').read_text()[-5000:])


{
  "version": "2026-08-19-joa-r-residual-final-package-v1",
  "status": "complete",
  "runtime_seconds": 104.05624104800017,
  "python": "3.12.13",
  "validation": {
    "delta_bss": 1.7445664990329715,
    "ci_low": -0.7023254806357085,
    "ci_high": 4.174683376264977,
    "positive_probability": 0.9265,
    "anchor_bss": 897.6978760835142,
    "candidate_bss": 899.4424425825542,
    "r_anchor_bss": 887.6184303312829,
    "r_candidate_bss": 889.5965581349641,
    "scale": 0.05,
    "feature_count": 116
  },
  "configuration": {
    "anchor": "JOA F-Regime075",
    "group": "R",
    "scale": 0.05,
    "seeds": [
      17,
      42,
      777
    ],
    "feature_count": 116,
    "removed_unavailable_meta": [
      "joa_f_minus_shared",
      "joa_f_probability",
      "joa_shared_probability"
    ]
  },
  "submission": {
    "path": "/content/drive/MyDrive/LG_AIMERS/joa_r_residual_final/submit_JOA_R_residual_multiseed005.zip",
    "size_mib": 274.28554916381836,
    "files": 114,
    

In [14]:
report = Path('/content/JOA_R_Residual_Final_Build_Report.zip')
report_names = [
    'run_summary.json', 'deployable_validation.json',
    'deployable_seed_validation.csv', 'submission_smoke.log', 'colab_build.log',
]
with zipfile.ZipFile(report, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for name in report_names:
        path = BUILD_DIR / name
        if path.exists():
            archive.write(path, name)
print('다운로드:', FINAL_ZIP)
files.download(str(FINAL_ZIP))
files.download(str(report))


다운로드: /content/drive/MyDrive/LG_AIMERS/joa_r_residual_final/submit_JOA_R_residual_multiseed005.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>